# AlzDetect — Cross-Validated Baseline Comparison (peer-review)

Replaces single-split + best-of-N selection with **stratified 5-fold CV (mean ± std)** and fills the controlled baseline table, varying **only** the imbalance-handling on identical frozen MobileNetV2 (224px) features:

1. No balancing  2. Class weights  3. Random oversampling  4. SMOTE  5. Focal loss  6. **Fuzzy (ours)**

Fair-comparison design: backbone frozen, features extracted once (no leakage); **no augmentation** (a confound); resampling fit on the train fold only; identical Dense(8) projection for all rows; the fuzzy row swaps the final layer for the TSK fuzzy inference layer.

**Setup:** Add Input `alzheimermridataset` (legendahmed); Internet ON; GPU P100.

In [ ]:
!pip install -q scikit-fuzzy imbalanced-learn
import os, glob
cands = glob.glob('/kaggle/input/**/all image', recursive=True)
assert cands, 'Add the legendahmed/alzheimermridataset input.'
DATA_DIR = cands[0]; print('DATA_DIR =', DATA_DIR)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable(package='alzdetect', name='FuzzyLayer')
class FuzzyLayer(layers.Layer):
    def __init__(self, n_rules, n_classes, **kwargs):
        super().__init__(**kwargs); self.n_rules=int(n_rules); self.n_classes=int(n_classes)
    def build(self, s):
        d=int(s[-1])
        self.centers=self.add_weight(name='centers',shape=(self.n_rules,d),initializer=tf.keras.initializers.RandomNormal(stddev=1.0),trainable=True)
        self.log_sigma=self.add_weight(name='log_sigma',shape=(self.n_rules,d),initializer='zeros',trainable=True)
        self.consequent=self.add_weight(name='consequent',shape=(self.n_rules,self.n_classes),initializer='glorot_uniform',trainable=True)
        super().build(s)
    def call(self, x):
        x=tf.expand_dims(x,1); c=tf.expand_dims(self.centers,0); sig=tf.exp(tf.expand_dims(self.log_sigma,0))
        log_mu=-0.5*tf.reduce_sum(tf.square((x-c)/sig),axis=-1)
        firing=tf.nn.softmax(log_mu,axis=1)
        return tf.nn.softmax(tf.matmul(firing,self.consequent))
    def get_config(self):
        cfg=super().get_config(); cfg.update({'n_rules':self.n_rules,'n_classes':self.n_classes}); return cfg

def focal_loss(gamma=2.0):
    def loss(yt,yp):
        yp=tf.clip_by_value(yp,1e-7,1-1e-7); ce=-yt*tf.math.log(yp)
        return tf.reduce_sum(tf.pow(1-yp,gamma)*ce,axis=-1)
    return loss

In [ ]:
import re
import numpy as np
import cv2

IMG_SIZE, N_SPLITS, EPOCHS, BATCH, PROJ_DIM, N_RULES, SEED = 224, 5, 40, 32, 8, 16, 42
np.random.seed(SEED)
PREFIX={'mildDem':'Mild Dementia','moderateDem':'Moderate Dementia','verymildDem':'Very mild Dementia','nonDem':'Non Demented'}
CLASS_ORDER=['Mild Dementia','Moderate Dementia','Very mild Dementia','Non Demented']
cmap={c:i for i,c in enumerate(CLASS_ORDER)}

imgs,labels=[],[]
for fn in sorted(os.listdir(DATA_DIR)):
    m=re.match(r'^[a-zA-Z]+',fn)
    if m and PREFIX.get(m.group()):
        im=cv2.imread(os.path.join(DATA_DIR,fn))
        if im is not None:
            imgs.append(cv2.resize(im,(IMG_SIZE,IMG_SIZE))); labels.append(cmap[PREFIX[m.group()]])
X_img=np.array(imgs,dtype=np.uint8); y=np.array(labels)
print('Loaded', X_img.shape, {CLASS_ORDER[i]:int((y==i).sum()) for i in range(4)})

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model

inp=Input(shape=(IMG_SIZE,IMG_SIZE,3))
base=MobileNetV2(weights='imagenet',include_top=False,input_tensor=inp); base.trainable=False
feat=Model(inp, GlobalAveragePooling2D()(base.output))
print('Extracting frozen features ...')
F=feat.predict(X_img,batch_size=32,verbose=1)
import gc; del X_img, imgs; gc.collect()
print('features:', F.shape)

In [ ]:
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE, RandomOverSampler

def dense_head(d):
    i=Input(shape=(d,)); x=Dense(PROJ_DIM,activation='relu')(i); x=Dropout(0.3)(x)
    return Model(i, Dense(4,activation='softmax')(x))
def fuzzy_head(d):
    i=Input(shape=(d,)); x=Dense(PROJ_DIM,activation='relu')(i)
    return Model(i, FuzzyLayer(N_RULES,4)(x))

def fit_eval(Xtr,ytr,Xte,yte,method,seed):
    tf.keras.utils.set_random_seed(seed)
    # Standardize per fold (fit on train only) -> stable training, no fold collapse.
    sc=StandardScaler().fit(Xtr)
    Xtr=sc.transform(Xtr).astype('float32'); Xte=sc.transform(Xte).astype('float32')
    d=Xtr.shape[1]; cw=None; loss='categorical_crossentropy'; Xt,yt=Xtr,ytr
    if method=='No balancing': head=dense_head(d)
    elif method=='Class weights':
        head=dense_head(d); w=compute_class_weight('balanced',classes=np.arange(4),y=ytr); cw={i:w[i] for i in range(4)}
    elif method=='Random oversampling':
        head=dense_head(d); Xt,yt=RandomOverSampler(random_state=seed).fit_resample(Xtr,ytr)
    elif method=='SMOTE':
        head=dense_head(d); k=max(1,min(5,np.bincount(ytr).min()-1)); Xt,yt=SMOTE(random_state=seed,k_neighbors=k).fit_resample(Xtr,ytr)
    elif method=='Focal loss':
        head=dense_head(d); loss=focal_loss(2.0)
    elif method=='Fuzzy (ours)':
        head=fuzzy_head(d); k=max(1,min(5,np.bincount(ytr).min()-1)); Xt,yt=SMOTE(random_state=seed,k_neighbors=k).fit_resample(Xtr,ytr)
    Xt2,Xv,yt2,yv=train_test_split(Xt,yt,test_size=0.1,stratify=yt,random_state=seed)
    head.compile(optimizer=Adam(5e-4),loss=loss,metrics=['accuracy'])
    head.fit(Xt2,to_categorical(yt2,4),validation_data=(Xv,to_categorical(yv,4)),
             epochs=80,batch_size=BATCH,verbose=0,class_weight=cw,
             callbacks=[EarlyStopping(monitor='val_loss',patience=8,restore_best_weights=True)])
    yp=np.argmax(head.predict(Xte,verbose=0),axis=1)
    return accuracy_score(yte,yp), f1_score(yte,yp,average='macro')

methods=['No balancing','Class weights','Random oversampling','SMOTE','Focal loss','Fuzzy (ours)']
res={m:{'acc':[],'f1':[]} for m in methods}
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
for fold,(tr,te) in enumerate(skf.split(F,y)):
    print(f'\n=== Fold {fold+1}/{N_SPLITS} ===')
    for m in methods:
        a,f1=fit_eval(F[tr],y[tr],F[te],y[te],m,SEED+fold)
        res[m]['acc'].append(a); res[m]['f1'].append(f1)
        print(f'  {m:<20s} acc={a:.4f} macroF1={f1:.4f}')

In [ ]:
import pandas as pd
rows=[]
print(f"{'Method':<22s}{'Accuracy':<22s}{'Macro-F1'}")
print('-'*64)
for m in methods:
    a=np.array(res[m]['acc']); f=np.array(res[m]['f1'])
    print(f'{m:<22s}{a.mean():.3f} ± {a.std():.3f}        {f.mean():.3f} ± {f.std():.3f}')
    rows.append({'method':m,'acc_mean':a.mean(),'acc_std':a.std(),'f1_mean':f.mean(),'f1_std':f.std()})
df=pd.DataFrame(rows); df.to_csv('/kaggle/working/crossval_results.csv',index=False)
print('\nSaved /kaggle/working/crossval_results.csv'); df